# Imports and basic setup

In [17]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from typing import Any

# Loading existing api functions

In [18]:
from src.tools.fpl_api import (
    get_bootstrap_data,
    get_fixtures,
    get_player_news,
    get_live_gameweek_data,
    get_player_detail,
    get_manager_team_summary,
)

from src.services.fpl_service import FPLService

fpl_service = FPLService()

# Pull the full bootstrap data directly

In [19]:
bootstrap_raw = fpl_service.get_bootstrap_static()

players_df = pd.DataFrame(bootstrap_raw["elements"])
teams_df = pd.DataFrame(bootstrap_raw["teams"])
events_df = pd.DataFrame(bootstrap_raw["events"])

print(players_df.shape)
print(teams_df.shape)
print(events_df.shape)

players_df.head()

(825, 105)
(20, 21)
(38, 29)


,can_transact,can_select,chance_of_playing_next_round,chance_of_playing_this_round,code,cost_change_event,cost_change_event_fall,cost_change_start,cost_change_start_fall,price_change_percent,...,now_cost_rank_type,form_rank,form_rank_type,points_per_game_rank,points_per_game_rank_type,selected_rank,selected_rank_type,starts_per_90,clean_sheets_per_90,defensive_contribution_per_90
0,True,True,NaN,NaN,154561,0,0,5,-5,0,...,1,12,1,51,3,9,1,1.0,0.48,0.00
1,True,True,NaN,NaN,109745,0,0,-5,5,0,...,43,442,56,581,66,286,37,0.0,0.00,0.00
2,True,False,0.0,0.0,463748,0,0,0,0,0,...,53,460,66,598,75,373,54,0.0,0.00,0.00
3,True,True,NaN,NaN,551221,0,0,-1,1,0,...,90,429,49,570,60,362,52,0.0,0.00,0.00
4,True,True,75.0,100.0,226597,0,0,12,-12,0,...,1,2,1,2,1,5,1,1.0,0.58,9.31


# Build clean lookup tables

In [20]:
team_id_to_name = dict(zip(teams_df["id"], teams_df["name"]))
team_id_to_short = dict(zip(teams_df["id"], teams_df["short_name"]))

players_df["team_name"] = players_df["team"].map(team_id_to_name)
players_df["team_short"] = players_df["team"].map(team_id_to_short)

position_map = {
    1: "GK",
    2: "DEF",
    3: "MID",
    4: "FWD",
}
players_df["position"] = players_df["element_type"].map(position_map)

players_df["full_name"] = players_df["first_name"] + " " + players_df["second_name"]

players_df[[
    "id", "web_name", "full_name", "team_name", "position", "now_cost",
    "total_points", "form", "minutes", "selected_by_percent"
]].head()

,id,web_name,full_name,team_name,position,now_cost,total_points,form,minutes,selected_by_percent
0,1,Raya,David Raya Martín,Arsenal,GK,60,129,7.0,2790,34.6
1,2,Arrizabalaga,Kepa Arrizabalaga Revuelta,Arsenal,GK,40,0,0.0,0,0.4
2,3,Hein,Karl Hein,Arsenal,GK,40,0,0.0,0,0.2
3,4,Setford,Tommy Setford,Arsenal,GK,39,0,0.0,0,0.2
4,5,Gabriel,Gabriel dos Santos Magalhães,Arsenal,DEF,72,173,10.0,2165,43.0


# Clean player metrics for analysis

A lot of FPL values come as strings. Converting them to numerics

In [21]:
numeric_cols = [
    "now_cost",
    "total_points",
    "minutes",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "saves",
    "bonus",
    "bps",
    "influence",
    "creativity",
    "threat",
    "ict_index",
]

for col in numeric_cols:
    if col in players_df.columns:
        players_df[col] = pd.to_numeric(players_df[col], errors="coerce")

players_df["form"] = pd.to_numeric(players_df["form"], errors="coerce")
players_df["selected_by_percent"] = pd.to_numeric(players_df["selected_by_percent"], errors="coerce")

players_df["price"] = players_df["now_cost"] / 10.0

players_df[["web_name", "price", "form", "minutes", "selected_by_percent"]].head()

,web_name,price,form,minutes,selected_by_percent
0,Raya,6.0,7.0,2790,34.6
1,Arrizabalaga,4.0,0.0,0,0.4
2,Hein,4.0,0.0,0,0.2
3,Setford,3.9,0.0,0,0.2
4,Gabriel,7.2,10.0,2165,43.0


# Pull and prepare fixtures

In [22]:
fixtures_raw = fpl_service.get_fixtures()
fixtures_df = pd.DataFrame(fixtures_raw)

fixtures_df["team_h_name"] = fixtures_df["team_h"].map(team_id_to_name)
fixtures_df["team_a_name"] = fixtures_df["team_a"].map(team_id_to_name)

fixtures_df[[
    "id", "event", "team_h_name", "team_a_name",
    "team_h_difficulty", "team_a_difficulty", "finished"
]].head()

,id,event,team_h_name,team_a_name,team_h_difficulty,team_a_difficulty,finished
0,307,NaN,Man City,Crystal Palace,3,5,False
1,1,1.0,Liverpool,Bournemouth,3,4,True
2,2,1.0,Aston Villa,Newcastle,3,4,True
3,3,1.0,Brighton,Fulham,2,3,True
4,6,1.0,Spurs,Burnley,1,3,True


## Build team-centric future fixtures table

In [23]:
def build_team_fixtures(fixtures_df: pd.DataFrame, from_gw: int) -> pd.DataFrame:
    upcoming = fixtures_df[(fixtures_df["event"].notna()) & (fixtures_df["event"] >= from_gw)].copy()

    home_df = upcoming[[
        "event", "team_h", "team_a", "team_h_difficulty", "team_h_name", "team_a_name"
    ]].copy()
    home_df["team"] = home_df["team_h"]
    home_df["opponent"] = home_df["team_a"]
    home_df["is_home"] = True
    home_df["difficulty"] = home_df["team_h_difficulty"]
    home_df["team_name"] = home_df["team_h_name"]
    home_df["opponent_name"] = home_df["team_a_name"]

    away_df = upcoming[[
        "event", "team_a", "team_h", "team_a_difficulty", "team_a_name", "team_h_name"
    ]].copy()
    away_df.columns = ["event", "team_a", "team_h", "team_a_difficulty", "team_a_name", "team_h_name"]
    away_df["team"] = away_df["team_a"]
    away_df["opponent"] = away_df["team_h"]
    away_df["is_home"] = False
    away_df["difficulty"] = away_df["team_a_difficulty"]
    away_df["team_name"] = away_df["team_a_name"]
    away_df["opponent_name"] = away_df["team_h_name"]

    result = pd.concat([
        home_df[["event", "team", "team_name", "opponent", "opponent_name", "is_home", "difficulty"]],
        away_df[["event", "team", "team_name", "opponent", "opponent_name", "is_home", "difficulty"]],
    ], ignore_index=True)

    return result.sort_values(["team", "event"]).reset_index(drop=True)

team_fixtures_df = build_team_fixtures(fixtures_df, from_gw=31)
team_fixtures_df.head(20)

,event,team,team_name,opponent,opponent_name,is_home,difficulty
0,32.0,1,Arsenal,4,Bournemouth,True,3
1,33.0,1,Arsenal,13,Man City,False,5
2,34.0,1,Arsenal,15,Newcastle,True,3
3,35.0,1,Arsenal,10,Fulham,True,2
4,36.0,1,Arsenal,19,West Ham,False,2
5,37.0,1,Arsenal,3,Burnley,True,1
6,38.0,1,Arsenal,8,Crystal Palace,False,3
7,31.0,2,Aston Villa,19,West Ham,True,2
8,32.0,2,Aston Villa,16,Nott'm Forest,False,3
9,33.0,2,Aston Villa,17,Sunderland,True,2


# Compute next 3 / next 5 fixture scores for teams

In [24]:
def compute_team_fixture_scores(team_fixtures_df: pd.DataFrame, horizon: int = 3) -> pd.DataFrame:
    rows = []

    for team_id, group in team_fixtures_df.groupby("team"):
        group = group.sort_values("event").head(horizon)
        rows.append({
            "team": team_id,
            "team_name": group["team_name"].iloc[0],
            f"fixtures_next_{horizon}": len(group),
            f"avg_difficulty_next_{horizon}": group["difficulty"].mean(),
            f"fixture_list_next_{horizon}": list(
                zip(group["event"], group["opponent_name"], group["is_home"], group["difficulty"])
            )
        })

    return pd.DataFrame(rows).sort_values(f"avg_difficulty_next_{horizon}")

team_fixture_score_3 = compute_team_fixture_scores(team_fixtures_df, horizon=3)
team_fixture_score_5 = compute_team_fixture_scores(team_fixtures_df, horizon=5)

team_fixture_score_3.head(10)

,team,team_name,fixtures_next_3,avg_difficulty_next_3,fixture_list_next_3
1,2,Aston Villa,3,2.333333,"[(31.0, West Ham, True, 2), (32.0, Nott'm Fore..."
15,16,Nott'm Forest,3,2.333333,"[(31.0, Spurs, False, 3), (32.0, Aston Villa, ..."
5,6,Brighton,3,2.666667,"[(31.0, Liverpool, True, 3), (32.0, Burnley, F..."
4,5,Brentford,3,2.666667,"[(31.0, Leeds, False, 3), (32.0, Everton, True..."
11,12,Liverpool,3,2.666667,"[(31.0, Brighton, False, 3), (32.0, Fulham, Tr..."
14,15,Newcastle,3,2.666667,"[(31.0, Sunderland, True, 2), (32.0, Crystal P..."
19,20,Wolves,3,2.666667,"[(32.0, West Ham, False, 2), (33.0, Leeds, Fal..."
7,8,Crystal Palace,3,3.000000,"[(32.0, Newcastle, True, 3), (33.0, West Ham, ..."
13,14,Man Utd,3,3.000000,"[(31.0, Bournemouth, False, 3), (32.0, Leeds, ..."
10,11,Leeds,3,3.000000,"[(31.0, Brentford, True, 3), (32.0, Man Utd, F..."


# Attach fixture scores to players

In [25]:
players_with_fixtures = players_df.merge(
    team_fixture_score_3[["team", "avg_difficulty_next_3", "fixtures_next_3"]],
    on="team",
    how="left"
).merge(
    team_fixture_score_5[["team", "avg_difficulty_next_5", "fixtures_next_5"]],
    on="team",
    how="left"
)

players_with_fixtures[[
    "web_name", "team_name", "position", "price", "form",
    "avg_difficulty_next_3", "avg_difficulty_next_5"
]].head()

,web_name,team_name,position,price,form,avg_difficulty_next_3,avg_difficulty_next_5
0,Raya,Arsenal,GK,6.0,7.0,3.666667,3.0
1,Arrizabalaga,Arsenal,GK,4.0,0.0,3.666667,3.0
2,Hein,Arsenal,GK,4.0,0.0,3.666667,3.0
3,Setford,Arsenal,GK,3.9,0.0,3.666667,3.0
4,Gabriel,Arsenal,DEF,7.2,10.0,3.666667,3.0


# Build a simple player scoring model

In [26]:
def min_max_scale(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    if s.max() == s.min():
        return pd.Series([0.0] * len(s), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

player_scores_df = players_with_fixtures.copy()

player_scores_df["minutes_per_match_proxy"] = player_scores_df["minutes"] / np.maximum(1, player_scores_df["starts"].fillna(1))
player_scores_df["value_per_million"] = player_scores_df["total_points"] / np.maximum(player_scores_df["price"], 0.1)

player_scores_df["form_score"] = min_max_scale(player_scores_df["form"])
player_scores_df["minutes_score"] = min_max_scale(player_scores_df["minutes"])
player_scores_df["value_score"] = min_max_scale(player_scores_df["value_per_million"])
player_scores_df["ownership_score"] = min_max_scale(player_scores_df["selected_by_percent"]).fillna(0)
player_scores_df["fixture_score"] = 1 - min_max_scale(player_scores_df["avg_difficulty_next_3"])

player_scores_df["composite_score"] = (
    0.30 * player_scores_df["form_score"] +
    0.20 * player_scores_df["minutes_score"] +
    0.20 * player_scores_df["value_score"] +
    0.30 * player_scores_df["fixture_score"]
)

player_scores_df = player_scores_df.sort_values("composite_score", ascending=False)

player_scores_df[[
    "web_name", "team_name", "position", "price", "form",
    "selected_by_percent", "avg_difficulty_next_3", "composite_score"
]].head(20)

,web_name,team_name,position,price,form,selected_by_percent,avg_difficulty_next_3,composite_score
623,Anderson,Nott'm Forest,MID,5.5,6.3,8.4,2.333333,0.876499
614,N.Williams,Nott'm Forest,DEF,4.7,7.0,3.0,2.333333,0.862416
621,Gibbs-White,Nott'm Forest,MID,7.4,6.3,5.0,2.333333,0.813970
611,Milenković,Nott'm Forest,DEF,5.1,5.0,3.2,2.333333,0.795601
542,B.Fernandes,Man Utd,MID,10.3,10.3,44.5,3.000000,0.791169
185,Kayode,Brentford,DEF,4.5,6.7,1.0,2.666667,0.782419
230,Van Hecke,Brighton,DEF,4.5,5.0,5.2,2.666667,0.778941
176,Kelleher,Brentford,GK,4.8,5.3,13.5,2.666667,0.774010
307,Richards,Crystal Palace,DEF,4.4,7.0,2.5,3.000000,0.750068
469,Szoboszlai,Liverpool,MID,7.0,6.3,14.1,2.666667,0.748256


# Tag template vs differential players

In [27]:
def ownership_bucket(x: float) -> str:
    if pd.isna(x):
        return "unknown"
    if x >= 25:
        return "template"
    if x >= 10:
        return "popular"
    if x >= 5:
        return "semi-differential"
    return "differential"

player_scores_df["ownership_bucket"] = player_scores_df["selected_by_percent"].apply(ownership_bucket)

player_scores_df[[
    "web_name", "selected_by_percent", "ownership_bucket"
]].sort_values("selected_by_percent", ascending=False).head(20)

,web_name,selected_by_percent,ownership_bucket
515,Haaland,54.9,template
485,Semenyo,53.6,template
291,João Pedro,50.3,template
542,B.Fernandes,44.5,template
4,Gabriel,43.0,template
209,Thiago,36.8,template
480,Ekitiké,35.5,template
455,Virgil,35.0,template
0,Raya,34.6,template
487,Guéhi,33.7,template


# Pull player news and create availability flags

In [28]:
news_df = players_df[[
    "id", "web_name", "team_name", "news", "status", "chance_of_playing_next_round"
]].copy()

news_df["news"] = news_df["news"].fillna("")
news_df["chance_of_playing_next_round"] = pd.to_numeric(
    news_df["chance_of_playing_next_round"], errors="coerce"
)

def availability_label(row):
    status = row["status"]
    chance = row["chance_of_playing_next_round"]

    if status == "a":
        return "available"
    if pd.notna(chance):
        if chance >= 75:
            return "minor-risk"
        if chance >= 25:
            return "major-risk"
        return "unlikely"
    return "flagged"

news_df["availability"] = news_df.apply(availability_label, axis=1)

news_df[news_df["availability"] != "available"].head(20)

,id,web_name,team_name,news,status,chance_of_playing_next_round,availability
2,3,Hein,Arsenal,Has joined Werder Bremen on loan for the rest ...,u,0.0,unlikely
4,5,Gabriel,Arsenal,Knee injury - 75% chance of playing,d,75.0,minor-risk
5,6,Saliba,Arsenal,Ankle injury - 75% chance of playing,d,75.0,minor-risk
7,8,J.Timber,Arsenal,Ankle injury - 75% chance of playing,d,75.0,minor-risk
8,9,Kiwior,Arsenal,has joined Porto on loan for the rest of the s...,u,0.0,unlikely
12,14,Kacurri,Arsenal,has joined Morecambe on loan for the rest of t...,u,0.0,unlikely
14,16,Saka,Arsenal,Knock - 75% chance of playing,d,75.0,minor-risk
15,17,Ødegaard,Arsenal,Knee injury - 75% chance of playing,d,75.0,minor-risk
16,18,Madueke,Arsenal,Knee injury - 75% chance of playing,d,75.0,minor-risk
18,20,Trossard,Arsenal,Hip injury - 75% chance of playing,d,75.0,minor-risk


# Merge player scores with news

In [29]:
player_model_df = player_scores_df.merge(
    news_df[["id", "status", "chance_of_playing_next_round", "availability", "news"]],
    on="id",
    how="left"
)

player_model_df[[
    "web_name", "team_name", "position", "price",
    "composite_score", "availability", "news"
]].head(20)

KeyError: "['news'] not in index"